# Marswalk — rock-detector training (RTX 4050 laptop)

**Status: UNTESTED ON REAL DATA. Ready to run; outputs are not trustworthy yet.**

This notebook trains a rock **detector** (torchvision Faster R-CNN) wired to the
`marswalk.ml` modules: the annotation schema, the **spatial-block holdout**
split, and the **F1-by-size** detection metric. It runs end-to-end on *synthetic*
tiles so you can verify the machinery on any machine (even CPU), then you point
it at real AI4Mars-style labels.

### Read before trusting any number here
- **PRD gate (MARSWALK_CONTEXT.md §3):** the ML detector must not inform routing
  until the baseline planner passes *and* an **ablation** shows rocks actually
  change route decisions. Until then this is a *capability*, not a validated input.
- **All thresholds/params are PLACEHOLDERS.** The gate metric is **F1 for rocks
  ≥ 1.5 m** on the spatial holdout.
- **What to paste back** to the team after a real run (bottom of the notebook):
  peak VRAM, epoch time, and the **F1-by-size table** on the held-out blocks.
- The code logic (data→split→train step→eval) was verified on synthetic data
  (CPU). The *trained result on real data* is what remains unverified.

## 0. Setup

In [ ]:
# If marswalk isn't installed (`pip install -e .`), add src to the path:
import sys, pathlib
for _c in (pathlib.Path.cwd() / "src", pathlib.Path.cwd().parent / "src"):
    if (_c / "marswalk").exists() and str(_c) not in sys.path:
        sys.path.insert(0, str(_c)); break

import time
import numpy as np
import torch
import torchvision
from torch.utils.data import DataLoader, Dataset

from marswalk.ml.annotation import AnnotationSet, BBox, RockLabel, Tile
from marswalk.ml.splits import spatial_block_split, splits_are_disjoint
from marswalk.ml.metrics_det import precision_recall_by_size
from marswalk.ml.train import TrainConfig  # the owner-facing config object

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("device:", device, "| CUDA:", torch.cuda.get_device_name(0) if device.type == "cuda" else "none")

## 1. Config

Reuses `marswalk.ml.train.TrainConfig` plus a few notebook-only knobs.

In [ ]:
cfg = TrainConfig(
    annotations_json="",   # set to a real AnnotationSet JSON to use real data; "" -> synthetic
    tile_size=256,         # real tiles; synthetic uses IMG_SIZE below
    batch_size=2,          # 6 GB VRAM -> 2-4
    grad_accum=2,
    encoder="fasterrcnn_mobilenet_v3_large_fpn",
    epochs=20,
    lr=5e-3,
    amp=True,
    channels_last=True,
    block_size_m=50.0,     # spatial holdout block size
    test_fraction=0.3,
    seed=0,
)

# Notebook-only knobs:
PRETRAINED = True   # COCO-pretrained weights (needs internet the first time)
IMG_SIZE = 256      # synthetic tile size
SCORE_THR = 0.5     # detection score threshold at eval
IOU_THR = 0.5
cfg.model_dump()

## 2. Data

Synthetic tiles give you a signal to learn (a bright blob per rock) so the loop
is exercised without real imagery. **For real data**, build an `AnnotationSet`
from your AI4Mars-style labels and set each `Tile.path` to its image file — the
dataset below reads the image when `path` exists and falls back to synthetic
otherwise. Keep `origin_x_m/origin_y_m` real so the **spatial** split is meaningful.

In [ ]:
def synthetic_annotations(n_tiles=24, size=IMG_SIZE, px_size_m=0.25, seed=0) -> AnnotationSet:
    rng = np.random.default_rng(seed)
    tiles, grid = [], int(np.ceil(np.sqrt(n_tiles)))
    for i in range(n_tiles):
        gx, gy = i % grid, i // grid
        labels = []
        for _ in range(int(rng.integers(1, 6))):
            d = float(rng.uniform(4, 28))  # px diameter
            x, y = float(rng.uniform(0, size - d)), float(rng.uniform(0, size - d))
            labels.append(RockLabel(bbox=BBox(x=x, y=y, w=d, h=d)))
        tiles.append(Tile(id=f"t{i}", origin_x_m=gx * size * px_size_m,
                          origin_y_m=gy * size * px_size_m, px_size_m=px_size_m,
                          width=size, height=size, labels=labels))
    return AnnotationSet(tiles=tiles)


def render_synthetic(tile: Tile) -> np.ndarray:
    rng = np.random.default_rng(abs(hash(tile.id)) % (2**32))
    img = rng.normal(0.35, 0.05, (tile.height, tile.width)).astype(np.float32)
    yy, xx = np.mgrid[0:tile.height, 0:tile.width]
    for lab in tile.labels:
        b = lab.bbox
        cx, cy, r = b.x + b.w / 2, b.y + b.h / 2, max(b.w, b.h) / 2
        img += 0.6 * np.exp(-(((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * (r / 1.5) ** 2)))
    return np.clip(img, 0, 1)


def load_image(tile: Tile) -> np.ndarray:
    # Real image if tile.path exists, else synthetic. REAL DATA: ensure the
    # image is (tile.height, tile.width); resize upstream if not.
    if tile.path and pathlib.Path(tile.path).exists():
        from PIL import Image
        arr = np.asarray(Image.open(tile.path).convert("L"), dtype=np.float32) / 255.0
        return arr
    return render_synthetic(tile)

In [ ]:
class TileDataset(Dataset):
    def __init__(self, annotations: AnnotationSet, ids):
        self.tiles = [annotations.by_id(i) for i in ids]

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, idx):
        tile = self.tiles[idx]
        img = load_image(tile)
        img_t = torch.from_numpy(img).unsqueeze(0).repeat(3, 1, 1)  # 3xHxW in [0,1]
        boxes = [[l.bbox.x, l.bbox.y, l.bbox.x + l.bbox.w, l.bbox.y + l.bbox.h] for l in tile.labels]
        if boxes:
            target = {"boxes": torch.tensor(boxes, dtype=torch.float32),
                      "labels": torch.ones(len(boxes), dtype=torch.int64)}  # class 1 = rock
        else:
            target = {"boxes": torch.zeros((0, 4), dtype=torch.float32),
                      "labels": torch.zeros((0,), dtype=torch.int64)}
        return img_t, target, tile.id


def collate(batch):
    return tuple(zip(*batch, strict=True))

## 3. Load annotations and make a **spatial** holdout split

In [ ]:
if cfg.annotations_json:
    ann = AnnotationSet.load(cfg.annotations_json)
    print("loaded real annotations:", ann.label_counts())
else:
    ann = synthetic_annotations(seed=cfg.seed)
    print("using SYNTHETIC annotations (set cfg.annotations_json for real data)")

train_ids, test_ids = spatial_block_split(ann, cfg.block_size_m, cfg.test_fraction, cfg.seed)
assert splits_are_disjoint(ann, train_ids, test_ids, cfg.block_size_m), "spatial leakage!"
print(f"{len(ann.tiles)} tiles -> train {len(train_ids)} / test {len(test_ids)} (blocks disjoint)")

## 4. Model — Faster R-CNN (class 1 = rock)

In [ ]:
def build_model(pretrained=PRETRAINED, num_classes=2):
    w = "DEFAULT" if pretrained else None
    return torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn(
        weights=w, weights_backbone=w, num_classes=num_classes)

model = build_model().to(device)
if cfg.channels_last and device.type == "cuda":
    model = model.to(memory_format=torch.channels_last)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"model ready: {n_params:.1f}M params | pretrained={PRETRAINED}")

## 5. Train

AMP + gradient accumulation + `channels_last` (all gated on CUDA), logging
**peak VRAM** (`torch.cuda.max_memory_allocated`) and **epoch time** — the two
numbers the team needs for the 6 GB budget.

In [ ]:
def train(model, ann, train_ids, cfg):
    dl = DataLoader(TileDataset(ann, train_ids), batch_size=cfg.batch_size,
                    shuffle=True, collate_fn=collate)
    use_amp = cfg.amp and device.type == "cuda"
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=cfg.lr, momentum=0.9, weight_decay=5e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    cl = cfg.channels_last and device.type == "cuda"

    model.train()
    for epoch in range(cfg.epochs):
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        t0, running = time.time(), 0.0
        opt.zero_grad()
        for step, (imgs, targets, _ids) in enumerate(dl):
            imgs = [(im.to(device, memory_format=torch.channels_last) if cl else im.to(device)) for im in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = sum(model(imgs, targets).values()) / cfg.grad_accum
            scaler.scale(loss).backward()
            if (step + 1) % cfg.grad_accum == 0:
                scaler.step(opt); scaler.update(); opt.zero_grad()
            running += float(loss) * cfg.grad_accum
        dt = time.time() - t0
        peak = (torch.cuda.max_memory_allocated() / 1e9) if device.type == "cuda" else float("nan")
        print(f"epoch {epoch+1}/{cfg.epochs} | loss {running/max(len(dl),1):.3f} | {dt:.1f}s | peak VRAM {peak:.2f} GB")
    return model

# Tip: for a quick machinery check first, set cfg.epochs = 1.
model = train(model, ann, train_ids, cfg)

## 6. Evaluate on the holdout — **F1 by size** (gate: ≥ 1.5 m)

In [ ]:
def evaluate(model, ann, test_ids, score_thr=SCORE_THR, iou_thr=IOU_THR):
    model.eval()
    agg = {}
    px = ann.by_id(test_ids[0]).px_size_m
    with torch.no_grad():
        for tid in test_ids:
            tile = ann.by_id(tid)
            img = load_image(tile)
            img_t = torch.from_numpy(img).unsqueeze(0).repeat(3, 1, 1).to(device)
            pred = model([img_t])[0]
            keep = pred["scores"] >= score_thr
            pboxes = [BBox(x=float(b[0]), y=float(b[1]), w=float(b[2]-b[0]), h=float(b[3]-b[1]))
                      for b in pred["boxes"][keep].cpu()]
            pscores = [float(s) for s in pred["scores"][keep].cpu()]
            gts = [l.bbox for l in tile.labels]
            # per-tile matching, then aggregate counts (avoids cross-tile matches)
            res = precision_recall_by_size(pboxes, gts, px, iou_thr=iou_thr, scores=pscores)
            for k, v in res.items():
                a = agg.setdefault(k, {"tp": 0.0, "fp": 0.0, "fn": 0.0})
                a["tp"] += v["tp"]; a["fp"] += v["fp"]; a["fn"] += v["fn"]
    rows = {}
    for k, a in agg.items():
        p = a["tp"]/(a["tp"]+a["fp"]) if (a["tp"]+a["fp"]) else 0.0
        r = a["tp"]/(a["tp"]+a["fn"]) if (a["tp"]+a["fn"]) else 0.0
        f1 = 2*p*r/(p+r) if (p+r) else 0.0
        rows[k] = {"precision": round(p,3), "recall": round(r,3), "f1": round(f1,3), **a}
    return rows

results = evaluate(model, ann, test_ids)
print(f"{'size bin':>14} | {'P':>5} {'R':>5} {'F1':>5} | tp/fp/fn")
for k, v in results.items():
    print(f"{k:>14} | {v['precision']:.2f} {v['recall']:.2f} {v['f1']:.2f} | "
          f"{v['tp']:.0f}/{v['fp']:.0f}/{v['fn']:.0f}")
gate = results.get("[1.5,inf)m", {}).get("f1", 0.0)
print(f"\nGATE metric — F1 for rocks >= 1.5 m: {gate:.3f}")
print("NOTE: on SYNTHETIC data with few epochs this is meaningless; it is the "
      "number to report on REAL held-out blocks.")

## 7. What to paste back to the team

After a **real** run on the RTX 4050:

- [ ] **Peak VRAM** (GB) and **epoch time** (s) at `batch_size`/`grad_accum` used.
- [ ] **F1-by-size table** above, on the spatial holdout — especially **F1 ≥ 1.5 m**.
- [ ] Data provenance: AI4Mars (or source), licence, tile count, `px_size_m`.
- [ ] Any config changes from the defaults here.
- [ ] A note confirming train/test blocks were **spatially disjoint** (asserted above).

Then update `AI_LOG.md` (P2 row) with your name under *Reviewed by* and the real
results. This stays a *capability* until the ablation (P6) shows rock-aware cost
changes route decisions.